# MSR – RQ3: what happens to PRs that break a rule?

For agent PRs that violate a context-file rule: are they **rejected**, **fixed by a human before merge**, or **merged as is**?
And do reviewers **notice** the violation?

A number alone ("60% of violating PRs get merged") means little, so every rate is compared with **complying** PRs of the same rule.

1. Reuse the RQ1 pipeline (`fetch_data.ipynb`) – it gives the labeled (PR, rule) pairs
2. Outcome of each PR: merged / rejected / still open
3. Label the agent's **own** commits separately from human commits
4. Fate of violating PRs: rejected / fixed then merged / merged as is
5. Are violating PRs merged less often than complying ones?
6. Do reviewers mention the violation?
7. Sanity check of the mention regexes

In [1]:
import io
import json
import re
from contextlib import redirect_stdout

import numpy as np
import pandas as pd
from statsmodels.discrete.conditional_models import ConditionalLogit

## 1. Reuse the RQ1 pipeline

Same as in `rq2.ipynb`: runs every code cell of `fetch_data.ipynb` before its section "## 8. RQ1" (so the hand labels in `package_manager_violations.csv` are never overwritten).

After this cell we have `pr_rule`: one row per (PR, rule) with the RQ1 `label`, plus `prs`, `CHECKERS` and `AIDEV`.

In [2]:
with open("fetch_data.ipynb") as notebook_file:
    fetch_data = json.load(notebook_file)

for cell in fetch_data["cells"]:
    source = "".join(cell["source"])
    if source.startswith("## 8."):
        break
    if cell["cell_type"] == "code":
        with redirect_stdout(io.StringIO()):
            exec(source)

print(len(pr_rule), "(PR, rule) pairs from", pr_rule.id.nunique(), "agent PRs")

7254 (PR, rule) pairs from 5923 agent PRs


## 2. Outcome of every PR

- `merged`: has a `merged_at`
- `rejected`: closed without merge
- `open`: still open at the AIDev cutoff – no outcome yet, dropped from the outcome analysis

`not_applicable` pairs are dropped (same as RQ1).

In [3]:
pairs = pr_rule[pr_rule.label != "not_applicable"].rename(columns={"rules": "rule"}).copy()

pairs["outcome"] = np.select(
    [pairs.merged_at.notna(), pairs.state == "closed"],
    ["merged", "rejected"],
    default="open",
)

pd.crosstab([pairs.rule, pairs.label], pairs.outcome, margins=True)

outcome                   merged  open  rejected   All
rule            label                                 
changelog       complied     263    29        29   321
                violated      69    59        65   193
commit_format   complied    1287   164       342  1793
                violated      93    31       126   250
package_manager complied     155    31        58   244
                violated       8     3        12    23
required_tests  complied     498   141       146   785
                violated     140    38        87   265
All                         2513   496       865  3874

## 3. What did the agent itself submit?

The RQ1 label looks at **all** commits of the PR, so a human who added the missing test before merge hides the violation.
To see fixes we label the agent's own commits separately (`agent_label`) and compare with the final label.

This only works for tools that commit under a **bot account** (Copilot, Devin, Cursor, Jules).
Claude Code and Codex commit under the user's own account, so agent and human commits look the same.
For them `agent_label` = final label and `fixes_visible` = False.

Limitation: the PR **title** is the final title. If a human renamed it, we can't tell (affects `commit_format`).

In [4]:
BOT_AUTHORS = {
    "Copilot": "Copilot",
    "Devin": "devin-ai-integration[bot]",
    "Cursor": "cursoragent",
    "Google_Jules": "google-labs-jules[bot]",
}

details = pd.read_parquet(f"{AIDEV}/pr_commit_details.parquet", columns=["pr_id", "author", "message", "filename", "status"])
details = details[details.pr_id.isin(pairs.id)]
pr_agent = prs.set_index("id").agent
details = details[details.author == details.pr_id.map(pr_agent).map(BOT_AUTHORS)]  # agent's own commits only

agent_files = details.dropna(subset=["filename"]).groupby("pr_id").filename.apply(set)
agent_added = details[details.status == "added"].groupby("pr_id").filename.apply(set)
agent_messages = details.dropna(subset=["message"]).groupby("pr_id").message.apply(lambda msgs: list({m.split("\n")[0] for m in msgs}))


def label_agent_commits(pair):
    if pair.agent not in BOT_AUTHORS:
        return pair.label
    agent_only = pair.copy()
    agent_only["changed_files"] = agent_files.get(pair.id, set())
    agent_only["added_files"] = agent_added.get(pair.id, set())
    agent_only["commit_messages"] = agent_messages.get(pair.id, [])
    return CHECKERS[pair.rule](agent_only)


pairs["fixes_visible"] = pairs.agent.isin(BOT_AUTHORS)
pairs["agent_label"] = [label_agent_commits(pair) for _, pair in pairs.iterrows()]

pd.crosstab([pairs.rule, pairs.agent_label], pairs.label)

label                           complied  violated
rule            agent_label                       
changelog       complied             279         0
                not_applicable        16         5
                violated              26       188
commit_format   complied            1754         0
                violated              39       250
package_manager complied             151         0
                not_applicable        93         0
                violated               0        23
required_tests  complied             681         0
                not_applicable        71         4
                violated              33       261

Rows: what the agent submitted. Columns: final state of the PR.
`violated` → `complied` = someone else fixed it inside the PR.
`agent_label = not_applicable` means the agent itself never touched the relevant area (e.g. a human added the source change); those rows are left out below.

## 4. Fate of violating PRs

For every pair where **the agent** violated the rule:
- `rejected`: closed without merge
- `fixed_then_merged`: merged, and the final PR complies (a human added the test / changelog / ...)
- `merged_as_is`: merged with the violation still there
- `open`: no outcome yet

Split by `fixes_visible`: for Claude Code / Codex `fixed_then_merged` is always 0 by construction, not because nobody fixed anything.

In [5]:
violating = pairs[pairs.agent_label == "violated"].copy()
violating["fate"] = np.select(
    [violating.outcome == "open", violating.outcome == "rejected", violating.label == "complied"],
    ["open", "rejected", "fixed_then_merged"],
    default="merged_as_is",
)

FATES = ["rejected", "fixed_then_merged", "merged_as_is", "open"]
fate_counts = pd.crosstab([violating.rule, violating.fixes_visible], violating.fate).reindex(columns=FATES, fill_value=0)
fate_share = fate_counts.div(fate_counts.sum(axis=1), axis=0)

pd.concat({"share": fate_share, "prs": fate_counts}, axis=1).round(2)

share                                       \
fate                          rejected fixed_then_merged merged_as_is  open   
rule            fixes_visible                                                 
changelog       False             0.33              0.00         0.47  0.20   
                True              0.30              0.08         0.30  0.32   
commit_format   False             0.49              0.00         0.51  0.00   
                True              0.48              0.08         0.29  0.14   
package_manager False             1.00              0.00         0.00  0.00   
                True              0.50              0.00         0.36  0.14   
required_tests  False             0.25              0.00         0.66  0.08   
                True              0.31              0.11         0.42  0.16   

                                   prs                                      
fate                          rejected fixed_then_merged merged_as_is open  
rule            fixes_visible                                               
changelog       False                5                 0            7    3  
                True                60                16           59   64  
commit_format   False               19                 0           20    0  
                True               121                20           73   36  
package_manager False                1                 0            0    0  
                True                11                 0            8    3  
required_tests  False               15                 0           39    5  
                True                73                26           98   38

## 5. Are violating PRs merged less often?

Only closed PRs (merged or rejected). If reviewers care about the rule, violating PRs should be merged **less** than complying ones.

Pooled merge rates first, then a conditional logistic regression grouped by repo (same model as RQ2), so we only compare PRs **within the same repo**.
Repo matters a lot here: some repos merge almost everything, others almost nothing.

- `odds_ratio` < 1: violating PRs are less likely to be merged
- CI containing 1: no clear effect

In [6]:
closed = pairs[(pairs.outcome != "open") & (pairs.agent_label != "not_applicable")].copy()
closed["merged"] = closed.outcome == "merged"
closed["violated"] = closed.agent_label == "violated"

pd.concat({
    "merge_rate": closed.pivot_table(index="rule", columns="violated", values="merged", aggfunc="mean"),
    "prs": closed.pivot_table(index="rule", columns="violated", values="merged", aggfunc="count"),
}, axis=1).round(2)

merge_rate         prs      
violated             False True  False True 
rule                                        
changelog             0.90  0.56   263   147
commit_format         0.79  0.45  1595   253
package_manager       0.65  0.40   132    20
required_tests        0.75  0.65   552   251

In [7]:
def effect(rows, outcome, exposure="violated"):
    model = ConditionalLogit(
        endog=rows[outcome].astype(int),
        exog=rows[[exposure]].astype(int),
        groups=rows.repo,
    ).fit(disp=False)

    ci_low, ci_high = np.exp(model.conf_int().loc[exposure])

    return pd.Series({
        "prs": len(rows),
        "repos": rows.repo.nunique(),
        "odds_ratio": np.exp(model.params[exposure]),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": model.pvalues[exposure],
    })


merge_effect = pd.DataFrame({rule: effect(rows, "merged") for rule, rows in closed.groupby("rule")}).T
merge_effect.round(3)

/Users/peco/School/ESS/ess-msr/.venv/lib/python3.13/site-packages/statsmodels/discrete/conditional_models.py:303: ModelWarning: Dropped 10 groups and 66 observations for having no within-group variance
  super().__init__(endog, exog, missing=missing, **kwargs)
/Users/peco/School/ESS/ess-msr/.venv/lib/python3.13/site-packages/statsmodels/discrete/conditional_models.py:303: ModelWarning: Dropped 66 groups and 168 observations for having no within-group variance
  super().__init__(endog, exog, missing=missing, **kwargs)


/Users/peco/School/ESS/ess-msr/.venv/lib/python3.13/site-packages/statsmodels/discrete/conditional_models.py:303: ModelWarning: Dropped 34 groups and 58 observations for having no within-group variance
  super().__init__(endog, exog, missing=missing, **kwargs)
/Users/peco/School/ESS/ess-msr/.venv/lib/python3.13/site-packages/statsmodels/discrete/conditional_models.py:303: ModelWarning: Dropped 61 groups and 128 observations for having no within-group variance
  super().__init__(endog, exog, missing=missing, **kwargs)


,prs,repos,odds_ratio,ci_low,ci_high,p_value
changelog,410.0,28.0,0.339,0.151,0.763,0.009
commit_format,1848.0,135.0,0.192,0.132,0.281,0.000
package_manager,152.0,46.0,0.147,0.015,1.418,0.097
required_tests,803.0,110.0,0.898,0.551,1.466,0.668


Same per agent (pooled merge rate, violating vs complying). Small cells are noisy – look at `prs`.

In [8]:
pd.concat({
    "merge_rate": closed.pivot_table(index=["rule", "agent"], columns="violated", values="merged", aggfunc="mean"),
    "prs": closed.pivot_table(index=["rule", "agent"], columns="violated", values="merged", aggfunc="count"),
}, axis=1).round(2)

merge_rate          prs       
violated                          False True   False  True 
rule            agent                                      
changelog       Claude_Code        1.00  0.60    4.0   10.0
                Copilot            0.71  0.48   80.0   77.0
                Cursor             1.00  0.66    4.0   38.0
                Devin               NaN  0.56    NaN   16.0
                Google_Jules       1.00  1.00    2.0    4.0
                OpenAI_Codex       0.98  0.50  173.0    2.0
commit_format   Claude_Code        0.77  0.47  172.0   15.0
                Copilot            0.81  0.54  504.0  131.0
                Cursor             0.83  0.17  175.0   63.0
                Devin              0.72  0.55  163.0   20.0
                OpenAI_Codex       0.79  0.54  581.0   24.0
package_manager Claude_Code        0.97   NaN   30.0    NaN
                Copilot            0.63  0.56   27.0    9.0
                Cursor             0.53  0.30   15.0   10.0
                Devin              0.44   NaN   50.0    NaN
                OpenAI_Codex       1.00  0.00   10.0    1.0
required_tests  Claude_Code        0.79  0.67   92.0   27.0
                Copilot            0.74  0.63  328.0  134.0
                Cursor             0.44  0.63   25.0   35.0
                Devin              0.25  0.63   12.0   27.0
                Google_Jules        NaN  1.00    NaN    1.0
                OpenAI_Codex       0.85  0.78   95.0   27.0

## 6. Do reviewers mention the violation?

We collect all **human** feedback on the PR: reviews, PR comments and inline review comments with `user_type == "User"`,
excluding the PR's own author (for Claude Code / Codex that's the person running the agent).
Bot feedback (CodeRabbit, changeset-bot, ...) is counted separately.

A comment "mentions" the rule if it matches a keyword regex for that rule. These are rough – "test" shows up in many comments for other reasons –
so check the precision in section 7 before reporting a number.

In [9]:
MENTION_PATTERNS = {
    "required_tests": r"\b(?:add|write|missing|include|need|no|without)\b[^.\n]{0,30}\b(?:unit |integration )?tests?\b|\btest coverage\b",
    "changelog": r"\bchange ?log\b|\bchangeset\b",
    "commit_format": r"conventional commit|commit (?:message|title)|pr title|semantic (?:pr|commit|title)",
    "package_manager": r"package-lock|lock ?file|\b(?:npm|yarn|pnpm|bun|poetry|uv)\b",
}

pull_requests = pd.read_parquet(f"{AIDEV}/pull_request.parquet", columns=["id", "user", "html_url", "repo_url"])
pull_requests = pull_requests[pull_requests.id.isin(pairs.id)]
pr_author = pull_requests.set_index("id").user

reviews = pd.read_parquet(f"{AIDEV}/pr_reviews.parquet", columns=["pr_id", "user", "user_type", "body"])
comments = pd.read_parquet(f"{AIDEV}/pr_comments.parquet", columns=["pr_id", "user", "user_type", "body"])

inline = pd.read_parquet(f"{AIDEV}/pr_review_comments.parquet", columns=["pull_request_url", "user", "user_type", "body"])
api_url_to_id = dict(zip(pull_requests.repo_url + "/pulls/" + pull_requests.html_url.str.split("/").str[-1], pull_requests.id))
inline["pr_id"] = inline.pull_request_url.map(api_url_to_id)

feedback = pd.concat([reviews, comments, inline.drop(columns="pull_request_url")], ignore_index=True)
feedback = feedback[feedback.pr_id.isin(pairs.id) & feedback.body.fillna("").str.strip().ne("")]
feedback = feedback[feedback.user != feedback.pr_id.map(pr_author)]
feedback = feedback[~feedback.user.isin(BOT_AUTHORS.values())]  # the agent replying to itself
feedback["by_human"] = feedback.user_type == "User"

feedback_by_pr = {pr_id: rows for pr_id, rows in feedback.groupby("pr_id")}


def mentioned(pr_id, rule, human):
    rows = feedback_by_pr.get(pr_id)
    if rows is None:
        return False
    bodies = rows.loc[rows.by_human == human, "body"]
    return bodies.str.contains(MENTION_PATTERNS[rule], case=False, regex=True).any()


closed["has_human_feedback"] = closed.id.isin(feedback.loc[feedback.by_human, "pr_id"])
closed["human_mentioned"] = [mentioned(pr_id, rule, True) for pr_id, rule in zip(closed.id, closed.rule)]
closed["bot_mentioned"] = [mentioned(pr_id, rule, False) for pr_id, rule in zip(closed.id, closed.rule)]

closed.groupby(["rule", "violated"])[["has_human_feedback", "human_mentioned", "bot_mentioned"]].mean().round(3)

has_human_feedback  human_mentioned  bot_mentioned
rule            violated                                                    
changelog       False                  0.274            0.076          0.087
                True                   0.585            0.027          0.034
commit_format   False                  0.377            0.015          0.213
                True                   0.352            0.020          0.083
package_manager False                  0.250            0.038          0.136
                True                   0.350            0.200          0.350
required_tests  False                  0.598            0.094          0.306
                True                   0.466            0.028          0.068

**Read the table as "caught rate"**: `human_mentioned` in the `violated = True` rows = share of violating PRs where a human pointed at the rule.

We do **not** compare it with complying PRs: a PR that edits `CHANGELOG.md` or adds tests naturally gets comments about the changelog / tests,
so complying PRs are mentioned more for a reason that has nothing to do with catching a violation.

Among violating PRs: does being caught change the fate? (`fixes_visible` PRs only, so `fixed_then_merged` is meaningful.)

In [10]:
caught = violating.merge(closed[["id", "rule", "human_mentioned"]], on=["id", "rule"])
caught = caught[caught.fixes_visible]
pd.crosstab([caught.rule, caught.human_mentioned], caught.fate, normalize="index").reindex(columns=FATES[:3], fill_value=0).round(2)

fate                             rejected  fixed_then_merged  merged_as_is
rule            human_mentioned                                           
changelog       False                0.46               0.11          0.44
                True                 0.00               0.50          0.50
commit_format   False                0.56               0.10          0.35
                True                 1.00               0.00          0.00
package_manager False                0.67               0.00          0.33
                True                 0.25               0.00          0.75
required_tests  False                0.37               0.13          0.50
                True                 0.43               0.14          0.43

## 7. Sanity check: do the mention regexes catch real complaints?

Random human comments on **violating** PRs that matched. Read them and estimate how many really talk about the rule
(report this as mention precision, like the RQ1 regex check).

In [11]:
for rule in MENTION_PATTERNS:
    pr_ids = closed.loc[(closed.rule == rule) & closed.violated & closed.human_mentioned, "id"]
    bodies = feedback.loc[feedback.pr_id.isin(pr_ids) & feedback.by_human, "body"]
    bodies = bodies[bodies.str.contains(MENTION_PATTERNS[rule], case=False, regex=True)]
    print(f"\n===== {rule} ({len(bodies)} matching comments) =====")
    for body in bodies.sample(min(10, len(bodies)), random_state=0):
        print("-", " ".join(body.split())[:200])


===== required_tests (7 matching comments) =====
- You don't need to add a new test file, just amend `unit-test-function-lookup-resolution.cpp` with a `SLANG_CHECK()` to call getName on `func1`.
- I'll add these 1 by 1 so I can test them.
- Approving because this seems like a trivially-correct fix, and no tests broke.
- @copilot please add fixtures for all of these new rules. There is an existing directory in `apps/oxlint/fixtures/tsgolint` which has a directory for `no-floating-promises`. Create a new directory for 
- ## Review: Implementation needs improvements Thank you for working on implementing the `only-export-components` rule\! After reviewing the implementation against the original eslint-plugin-react-refre
- @copilot add test for `export * as default from "mod";`
- Closing this PR as we need to fix the tests first to be able to with the latest Spirv-Tools

===== changelog (5 matching comments) =====
- @copilot ```suggestion readme = {file = ["README.md", "CHANGELOG.md"], con